<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Databricks Performance Optimization
</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">06 Lab 1 - Layout</span>
    </div>
</div>

![DB Academy](https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/common/databricks_academy_logo.png)

# Lab - Layout
#### Duration: 20 minutes

## Overview

John Smith wrote a usage report for the CloudField SaaS platform two years ago. It still returns the right answer, but it has gotten slow and it runs every morning.

Your job is to make it efficient. This is a lab, so you diagnose and fix it yourself.

You will:

- Measure the inherited report and record an honest baseline.
- Read the layout evidence and name the root cause.
- Re-lay out the table with liquid clustering, then prove the report reads less and costs less.

Everything you need was covered in Parts 02 through 05: measure first, data layout, liquid clustering and honest benchmarking. 

Each step in Section E has a Copy button with the exact code if you get stuck.

> Serverless-first: this lab runs entirely on a serverless notebook. The lab data lives in your own `perf_lab` schema and does not touch the demo data.

## Learning Objectives

By the end of this lab, you will be able to:

1. **Open a Query Profile** for an inherited slow report and name the dominant bottleneck.
2. **Diagnose over-partitioning** and stats-poor, non-selective files as the root cause from the layout evidence.
3. **Re-lay out the table** with liquid clustering via CTAS, choosing keys from the columns the report filters on.
4. **Re-benchmark honestly** with the result cache off, a discarded warm-up, and a steady-state median.
5. **Confirm the win** with pruning evidence, `pruned_files` rising above zero, in `system_vw.query.history`.

<div style="border-left:4px solid #FFAB00;background:rgba(255,171,0,0.08);padding:14px 18px;border-radius:4px;margin:16px 0;font-family:sans-serif;">
  <strong style="display:block;color:#B8860B;margin-bottom:6px;font-size:15pt;">Pre-Run Timing Note</strong>
  <div style="color:#0b2026;font-size:14pt;line-height:1.55;">
    This notebook reads from <code>system_vw.query.history</code>, which lags live activity by <strong>5 to 10 minutes</strong>. Run the query cells first, then continue with the explanatory sections. By the time you reach the system-table cells, the rows will have landed. If a cell returns no rows, wait a few minutes and re-run it.
  </div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**

![Serverless Select](https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:**  This notebook requires **Python and was tested using Serverless V5**. The whole lab, including the CTAS re-layout, runs on serverless. Other compute may work but is not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>

## A. Classroom Setup

Run the setup below. It:
- builds the inherited **usage_events** table in your own **perf_lab** schema (SaaS usage telemetry, isolated from the demo data)
- runs `ANALYZE` so the before and after comparison is fair
- defines the report plus the `benchmark_query`, `measure_layout` and `prove_pruning` helpers you will use.

In [ ]:
%run ./Includes/Classroom-Setup-06

## B. The Scenario
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.blue{--accent:#4299E0;} .green{--accent:#00A972;} .coral{--accent:#FF5F46;} .amber{--accent:#FFAB00;} .red{--accent:#98102A;}
.ds-scenario { background: #F9F7F4; border-radius: 10px; overflow: hidden; box-shadow: 0 2px 8px rgba(27,49,57,0.06); }
.ds-scenario-band { background: var(--accent); color: #fff; padding: 14px 20px; text-align: center; font-size: 18pt; font-weight: 700; }
.ds-scenario-body { padding: 18px 20px; }
.ds-scenario-q { font-size: 16pt; font-style: italic; color: #0b2026; margin-bottom: 12px; text-align: center; }
.ds-scenario-text { font-size: 14pt; color: #0b2026; line-height: 1.6; }
.ds-scenario-text ul { margin: 12px 0; padding-left: 18px; }
.ds-scenario-text li { margin-bottom: 8px; }
</style>

<div class="ds-row">
  <div class="ds-scenario coral">
    <div class="ds-scenario-band">John Smith's morning usage report</div>
    <div class="ds-scenario-body">
      <div class="ds-scenario-q">"It worked fine two years ago. Now the account team waits on it every morning.."</div>
      <div class="ds-scenario-text">The report pulls one enterprise account's full usage history from the <b>usage_events</b> table and rolls it up by feature and event type.
        <ul>
          <li>The business still runs it daily to review a top account before the account sync.</li>
          <li>The table has grown to millions of events across two years of daily activity.</li>
          <li>Nobody has touched the layout since it was created. It was set up the way tables "used to be" set up.</li>
        </ul>
        <strong>Your task: measure it, find the root cause, re-lay out the data and show the report got cheaper.</strong>
      </div>
    </div>
  </div>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### How to work this lab
- Treat every section as a task, not a walkthrough. Read the prompt, then do the work in the code cells.
- Change one thing at a time. That is the only way to know which change helped.
- Use the tools from the earlier parts: the Query Profile, `DESCRIBE DETAIL`, and `system_vw.query.history`.
- The setup already ran `ANALYZE` on the inherited table, so the only variable you change is the physical layout.
- If you get stuck, use the Copy buttons in Section E, then run Section F to confirm the win.

</details>

### The report you inherited

<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">
  <pre style="background:#F9F7F4;border:1px solid #EEEDE9;border-radius:8px;padding:16px;overflow-x:auto;margin:0;font-family:Menlo,Consolas,monospace;font-size:13pt;line-height:1.5;color:#0b2026;">
SELECT
  feature,
  event_type,
  COUNT(*) AS events,
  ROUND(SUM(usage_units), 2) AS total_units,
  COUNT(DISTINCT event_date) AS active_days
FROM `&lt;my_catalog&gt;`.`perf_lab`.usage_events
WHERE account_key = 7
GROUP BY feature, event_type
ORDER BY total_units DESC
</pre>
</div>

The setup exposes this as `BEFORE_REPORT_SQL` (and `report_sql(table)` if you want to run it against any table). You do not need to edit the report itself. The report is fine. The data layout underneath it is the problem.

## C. Measure First

Before changing anything, measure the current state and record an honest baseline. This is the same loop from Part 02.

**Your task:**
1. Run the cell below. It executes the report once (so a Query Profile exists) and then benchmarks a steady-state median.
2. Open the Query Profile: click **See performance** under the cell, or open **Query History** and choose **See query profile**.
3. Record the baseline: the median wall time printed here, plus **files read**, **files pruned**, and **bytes read** from the Scan operator in the profile.

In [ ]:
# Run the report once so its Query Profile exists. 
before_report_df = spark.sql(BEFORE_REPORT_SQL)
before_report_rows = before_report_df.count()
print(f"Report returned {before_report_rows:,} grouped rows for account_key = {DEMO_ACCOUNT_KEY}.")
print("Open the Query Profile now: 'See performance' under this cell, or Query History -> See query profile.\n")

# The baseline: discard a warm-up, report the median (result cache is bypassed).
median_before = benchmark_query(BEFORE_REPORT_SQL, "before (partitioned)")

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### What to record from the profile, and why
- **Median wall time**: the number printed above. It is the median of several runs with the result cache bypassed and the warm-up discarded. That is the honest latency, not a lucky cache hit.
- **Files read vs files pruned** (Scan operator): the single most damning pair on this report. A filter on one account out of tens of thousands should skip almost the whole table. Watch what it actually skips.
- **Bytes read**: how much data the scan touched to answer a question about one small account.
- **Two clocks**: aggregated task time much larger than wall time means the work ran in parallel across many tasks. Lots of parallel work to find one account's rows is the tell that the scan is reading far too much.
- You are not fixing anything yet. Naming the bottleneck from the profile is the skill here. The fix comes after you have diagnosed the cause in Section D.
- `system_vw.query.history` has a 5 to 10 minute ingestion lag, so the immediate signal is the Query Profile you opened by hand. Section F pulls the same numbers from the system table for the before and after comparison.

</details>

## D. Diagnose the Root Cause
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; display: flex; flex-direction: column; gap: 20px; }
.cw * { box-sizing: border-box; }
.ds-banner { display: flex; align-items: center; gap: 20px; width: 100%; border-radius: 10px; padding: 24px 28px; background: var(--bnr-bg); border-left: 8px solid var(--bnr-accent); box-shadow: 0 2px 10px rgba(27,49,57,0.08); }
.ds-banner-icon { width: 60px; height: 60px; border-radius: 50%; flex-shrink: 0; background: var(--bnr-accent); color: var(--bnr-icon-fg); font-size: 30pt; font-weight: 700; line-height: 1; display: flex; align-items: center; justify-content: center; }
.ds-banner-title { font-size: 22pt; font-weight: 700; color: var(--bnr-title); line-height: 1.2; margin-bottom: 4px; }
.ds-banner-tagline { font-size: 15pt; color: var(--bnr-tagline); line-height: 1.4; }
.ds-banner.caution { --bnr-bg: rgba(255,171,0,0.12); --bnr-accent: #FFAB00; --bnr-icon-fg: #0b2026; --bnr-title: #0b2026; --bnr-tagline: #5E7077; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 260px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.blue{--accent:#4299E0;} .green{--accent:#00A972;} .coral{--accent:#FF5F46;} .amber{--accent:#FFAB00;} .red{--accent:#98102A;}
.ds-card-label { font-size: 14pt; font-weight: 700; letter-spacing: 2px; text-transform: uppercase; color: var(--accent); margin-bottom: 4px; }
.ds-card-title { font-size: 18pt; font-weight: 700; line-height: 1.25; margin: 0 0 8px; }
.ds-card ul { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-card li { margin-bottom: 10px; }
.ds-card li:last-child { margin-bottom: 0; }
</style>

<div class="ds-banner caution">
  <div class="ds-banner-icon">?</div>
  <div>
    <div class="ds-banner-title">Ask the layout three questions</div>
    <div class="ds-banner-tagline">The profile told you the scan reads too much. Now find out why. Inspect the layout in the next cell, then answer these.</div>
  </div>
</div>

<div class="ds-row">
  <div class="ds-card amber">
    <div class="ds-card-label">Question 1</div>
    <div class="ds-card-title">How is the table laid out?</div>
    <ul>
      <li>Is it partitioned? On which column?</li>
      <li>How many files does it have, and how big is the average file?</li>
      <li>Does the file count look proportional to the data size, or far larger?</li>
    </ul>
  </div>
  <div class="ds-card coral">
    <div class="ds-card-label">Question 2</div>
    <div class="ds-card-title">Does the layout match the filter?</div>
    <ul>
      <li>Which column does the report filter on?</li>
      <li>Is that the same column the table is partitioned on?</li>
      <li>If not, can partition pruning help this report at all?</li>
    </ul>
  </div>
  <div class="ds-card red">
    <div class="ds-card-label">Question 3</div>
    <div class="ds-card-title">Why is nothing pruned?</div>
    <ul>
      <li>For one account's rows to be skippable, they must be co-located in a few files.</li>
      <li>If the account's rows are scattered across every file, each file's min and max for `account_key` spans the filter value.</li>
      <li>What layout would co-locate one account's rows?</li>
    </ul>
  </div>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### The diagnosis, once you have the evidence
- **Over-partitioning.** The table was partitioned by `event_date`, one physical folder per day, across two years. That is hundreds of partitions on a table only a few hundred MB in size, so each partition is a tiny file, well under the 1 GB per partition floor. This is the partition-by-date reflex from an older era.
- **The layout does not match the filter.** The report filters on `account_key`, not on `event_date`. Partition pruning only fires when you filter the partition column, so it does nothing here. The scan opens every daily partition.
- **Nothing prunes at the file level either.** Because a single account is active on many days, its rows are scattered across every date file. Every file's min and max for `account_key` includes the filter value, so data skipping cannot eliminate a single file. `pruned_files` stays at 0.
- **The fix is to lay the data out by what the report filters on.** Cluster the table by `account_key` so one account's rows are co-located into a small region of the table. Then the scan can skip the files that do not contain that account.
- This is exactly the partitioned-versus-clustered contrast from Part 04, now as something you have to spot yourself from an inherited table.

</details>

In [ ]:
# measure_layout() reads numFiles, size, partitionColumns and clusteringColumns from DESCRIBE DETAIL.
measure_layout(PARTITIONED_TABLE, "usage_events (inherited)")

# How many partitions does the over-partitioning create
_n_partitions = spark.sql(f"SHOW PARTITIONS {PARTITIONED_TABLE}").count()
print(f"\nusage_events has {_n_partitions:,} partitions (one per event_date).")
print(f"The report filters on account_key, not event_date, so none of those partitions can be pruned.")

<div style="width: 100%; font-family: sans-serif;">
<div style="background: #F9F7F4; border-radius: 10px; padding: 24px 28px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); border-top: 6px solid #FF5F46;">
  <div style="margin-bottom: 10px;"><svg viewBox="0 0 806 200" fill="none" xmlns="http://www.w3.org/2000/svg" style="height:56px; width:auto; display:block;"><path d="M81.7909 159.789C75.0244 159.789 69.5392 166.599 69.5392 175H126.368C126.368 166.599 120.883 159.789 114.116 159.789H81.7909Z" fill="#FF5F46"/><path d="M179.614 80.7871C167.525 100.208 149.376 126.933 141.272 133.79C133.167 140.647 121.149 153.009 93.2907 153.009C73.7741 153.009 56.9864 140.474 49.5352 122.498C49.5353 122.519 49.5356 122.54 49.5358 122.561C48.4799 119.938 45.9132 118.085 42.9121 118.085C38.969 118.085 35.7721 121.282 35.7721 125.225C35.7722 126.15 35.9484 126.997 36.2689 127.752C37.984 131.793 42.5469 131.965 42.5469 144.569C42.5469 143.103 42.5469 146.404 42.5469 144.569C30.6947 144.569 21.0865 134.96 21.0863 123.108C21.0863 111.256 30.6945 101.647 42.5469 101.647H45.4249V101.644H79.5153C82.062 101.644 84.5067 102.722 86.0978 104.71C88.8856 108.194 91.0222 111.438 92.5969 114.114C94.056 116.594 98.4255 116.618 99.8699 114.13C101.393 111.506 103.416 108.337 105.98 104.929C107.573 102.813 110.103 101.644 112.752 101.644H125.427C143.951 101.644 154.105 92.7623 159.289 86.3612C161.972 83.049 165.782 80.7871 170.045 80.7871H179.614Z" fill="#FABFBA"/><path d="M96.3843 111.768C96.3842 87.8629 77.3194 68.4739 53.768 68.3843C77.3196 68.2949 96.3845 48.9049 96.3845 25C96.3845 25 96.3845 25 96.3845 25C96.3845 48.9602 115.538 68.3838 139.165 68.3838C115.538 68.3838 96.3844 87.8075 96.3843 111.768Z" fill="#FF5F46"/><path d="M274.824 139.867C267.307 139.867 260.701 138.158 255.006 134.742C249.388 131.325 245.022 126.579 241.909 120.505C238.796 114.355 237.239 107.18 237.239 98.9793C237.239 90.855 238.834 83.6798 242.023 77.4537C245.212 71.1516 249.767 66.2543 255.69 62.7615C261.688 59.1929 268.787 57.4086 276.988 57.4086C286.251 57.4086 293.92 59.6485 299.994 64.1283C306.144 68.5321 310.13 74.6823 311.952 82.5788H296.577C295.438 78.6306 293.198 75.5555 289.857 73.3535C286.516 71.1516 282.189 70.0507 276.874 70.0507C271.483 70.0507 266.851 71.2276 262.979 73.5813C259.182 75.8592 256.259 79.2 254.209 83.6039C252.235 87.9318 251.248 93.0949 251.248 99.0932C251.248 105.092 252.235 110.255 254.209 114.583C256.259 118.911 259.144 122.213 262.865 124.491C266.585 126.693 270.951 127.794 275.962 127.794C283.403 127.794 289.022 125.706 292.819 121.53C296.615 117.354 298.855 111.925 299.538 105.243H278.924V94.8792H313.319V138.5H300.791L299.652 127.566C297.906 130.148 295.894 132.388 293.616 134.286C291.338 136.108 288.642 137.513 285.529 138.5C282.492 139.411 278.924 139.867 274.824 139.867ZM351.07 139.867C345.375 139.867 340.364 138.652 336.036 136.222C331.708 133.717 328.33 130.224 325.9 125.744C323.47 121.264 322.255 116.101 322.255 110.255C322.255 104.18 323.432 98.8654 325.786 94.3097C328.216 89.7541 331.594 86.1854 335.922 83.6039C340.326 81.0223 345.413 79.7315 351.184 79.7315C356.803 79.7315 361.7 80.9843 365.876 83.49C370.052 85.9956 373.279 89.3744 375.557 93.6264C377.835 97.8024 378.974 102.51 378.974 107.749C378.974 108.508 378.974 109.344 378.974 110.255C378.974 111.166 378.898 112.115 378.746 113.102H332.05V104.332H365.193C365.041 100.156 363.636 96.8913 360.979 94.5375C358.321 92.1078 355.018 90.893 351.07 90.893C348.261 90.893 345.679 91.5384 343.325 92.8291C340.972 94.1199 339.111 96.0561 337.745 98.6377C336.378 101.143 335.695 104.332 335.695 108.205V111.507C335.695 115.076 336.34 118.151 337.631 120.733C338.997 123.314 340.82 125.288 343.098 126.655C345.451 127.946 348.071 128.591 350.956 128.591C354.145 128.591 356.765 127.908 358.815 126.541C360.941 125.175 362.497 123.352 363.484 121.074H377.379C376.316 124.643 374.57 127.87 372.14 130.755C369.71 133.565 366.711 135.805 363.143 137.475C359.574 139.069 355.55 139.867 351.07 139.867ZM388.176 138.5V81.0982H400.248L401.273 90.6652C403.02 87.3243 405.525 84.6669 408.79 82.6927C412.055 80.7186 415.928 79.7315 420.407 79.7315C425.039 79.7315 428.987 80.7186 432.252 82.6927C435.517 84.5909 438.023 87.4003 439.769 91.1208C441.591 94.8412 442.503 99.4729 442.503 105.016V138.5H428.835V106.268C428.835 101.485 427.772 97.8024 425.646 95.2209C423.52 92.6393 420.369 91.3485 416.193 91.3485C413.46 91.3485 410.992 91.9939 408.79 93.2847C406.664 94.5755 404.956 96.4737 403.665 98.9793C402.45 101.409 401.843 104.37 401.843 107.863V138.5H388.176ZM454.064 138.5V81.0982H467.731V138.5H454.064ZM460.898 71.9868C458.392 71.9868 456.304 71.2275 454.634 69.709C453.039 68.1145 452.242 66.1783 452.242 63.9005C452.242 61.5467 453.039 59.6485 454.634 58.2059C456.304 56.6873 458.392 55.928 460.898 55.928C463.404 55.928 465.454 56.6873 467.048 58.2059C468.719 59.6485 469.554 61.5467 469.554 63.9005C469.554 66.1783 468.719 68.1145 467.048 69.709C465.454 71.2275 463.404 71.9868 460.898 71.9868ZM506.987 139.867C501.292 139.867 496.281 138.652 491.953 136.222C487.625 133.717 484.246 130.224 481.817 125.744C479.387 121.264 478.172 116.101 478.172 110.255C478.172 104.18 479.349 98.8654 481.703 94.3097C484.133 89.7541 487.511 86.1854 491.839 83.6039C496.243 81.0223 501.33 79.7315 507.101 79.7315C512.72 79.7315 517.617 80.9843 521.793 83.49C525.969 85.9956 529.196 89.3744 531.474 93.6264C533.752 97.8024 534.891 102.51 534.891 107.749C534.891 108.508 534.891 109.344 534.891 110.255C534.891 111.166 534.815 112.115 534.663 113.102H487.967V104.332H521.11C520.958 100.156 519.553 96.8913 516.896 94.5375C514.238 92.1078 510.935 90.893 506.987 90.893C504.178 90.893 501.596 91.5384 499.242 92.8291C496.889 94.1199 495.028 96.0561 493.662 98.6377C492.295 101.143 491.612 104.332 491.612 108.205V111.507C491.612 115.076 492.257 118.151 493.548 120.733C494.914 123.314 496.737 125.288 499.015 126.655C501.368 127.946 503.988 128.591 506.873 128.591C510.062 128.591 512.682 127.908 514.732 126.541C516.858 125.175 518.414 123.352 519.401 121.074H533.296C532.233 124.643 530.487 127.87 528.057 130.755C525.627 133.565 522.628 135.805 519.06 137.475C515.491 139.069 511.467 139.867 506.987 139.867Z" fill="#0B2026"/><rect x="563.157" y="58.4648" width="242.843" height="84.9055" rx="42.4527" fill="#0B2026"/><path d="M664.499 114.947C661.815 114.947 659.493 114.366 657.532 113.205C655.596 112.018 654.1 110.38 653.042 108.289C652.01 106.173 651.493 103.735 651.493 100.974C651.493 98.187 652.01 95.7485 653.042 93.6584C654.1 91.5424 655.596 89.8909 657.532 88.7039C659.493 87.517 661.815 86.9235 664.499 86.9235C667.724 86.9235 670.356 87.7234 672.395 89.3232C674.433 90.8973 675.724 93.1294 676.266 96.0195H671.156C670.795 94.497 670.06 93.2971 668.95 92.4198C667.84 91.5424 666.344 91.1037 664.46 91.1037C662.757 91.1037 661.286 91.5037 660.048 92.3036C658.835 93.0778 657.893 94.2132 657.222 95.7098C656.577 97.1807 656.254 98.9353 656.254 100.974C656.254 103.012 656.577 104.767 657.222 106.238C657.893 107.683 658.835 108.805 660.048 109.605C661.286 110.38 662.757 110.767 664.46 110.767C666.344 110.767 667.84 110.367 668.95 109.567C670.06 108.741 670.795 107.606 671.156 106.161H676.266C675.749 108.896 674.459 111.05 672.395 112.625C670.356 114.173 667.724 114.947 664.499 114.947ZM697.803 114.947C695.196 114.947 692.887 114.366 690.874 113.205C688.887 112.018 687.326 110.38 686.191 108.289C685.055 106.173 684.488 103.722 684.488 100.935C684.488 98.1741 685.055 95.7485 686.191 93.6584C687.326 91.5424 688.887 89.8909 690.874 88.7039C692.887 87.517 695.196 86.9235 697.803 86.9235C700.46 86.9235 702.796 87.517 704.808 88.7039C706.821 89.8909 708.382 91.5424 709.492 93.6584C710.602 95.7485 711.156 98.1741 711.156 100.935C711.156 103.722 710.602 106.173 709.492 108.289C708.382 110.38 706.821 112.018 704.808 113.205C702.796 114.366 700.46 114.947 697.803 114.947ZM697.841 110.767C699.57 110.767 701.08 110.38 702.37 109.605C703.66 108.805 704.654 107.67 705.35 106.199C706.047 104.728 706.395 102.974 706.395 100.935C706.395 98.8966 706.047 97.1419 705.35 95.6711C704.654 94.2003 703.66 93.0778 702.37 92.3036C701.08 91.5037 699.57 91.1037 697.841 91.1037C696.112 91.1037 694.603 91.5037 693.313 92.3036C692.022 93.0778 691.016 94.2003 690.294 95.6711C689.597 97.1419 689.249 98.8966 689.249 100.935C689.249 102.974 689.597 104.728 690.294 106.199C691.016 107.67 692.022 108.805 693.313 109.605C694.603 110.38 696.112 110.767 697.841 110.767ZM720.339 114.482V87.3879H729.087C732.235 87.3879 734.829 87.9427 736.867 89.0523C738.931 90.1619 740.454 91.7359 741.434 93.7745C742.441 95.7872 742.944 98.187 742.944 100.974C742.944 103.709 742.441 106.096 741.434 108.135C740.454 110.147 738.931 111.708 736.867 112.818C734.829 113.928 732.235 114.482 729.087 114.482H720.339ZM724.984 110.534H728.855C731.203 110.534 733.048 110.16 734.39 109.412C735.757 108.638 736.725 107.541 737.293 106.122C737.886 104.677 738.183 102.961 738.183 100.974C738.183 98.9611 737.886 97.2452 737.293 95.8259C736.725 94.3809 735.757 93.2713 734.39 92.4972C733.048 91.6972 731.203 91.2973 728.855 91.2973H724.984V110.534ZM752.124 114.482V87.3879H769.619V91.1425H756.769V98.9224H768.458V102.6H756.769V110.728H769.619V114.482H752.124Z" fill="white"/><path d="M636.472 101.239L620.99 116.72L613.554 109.283L621.598 101.239L613.554 93.1943L620.99 85.7578L636.472 101.239Z" fill="#FF3621"/><path d="M584.879 101.241L600.36 85.7598L607.797 93.1968L599.753 101.241L607.797 109.286L600.36 116.723L584.879 101.241Z" fill="#FF3621"/></svg></div>
  <div style="font-size: 15pt; color: #0b2026; line-height: 1.7; margin-bottom: 16px;">You diagnosed this yourself, which is the point. Now compare your reasoning against <strong>Genie Code</strong>. Ask it why the report scans so many files, then judge whether its answer matches the evidence you gathered. Click <strong>Copy</strong> and paste the prompt into Genie Code in the notebook or SQL editor.</div>
  <div style="display: flex; align-items: center; gap: 10px; background: #fff; border: 1px solid #EEEDE9; border-radius: 6px; padding: 10px 14px; font-size: 14pt; font-family: monospace; color: #0b2026;">
    <span id="genie-query" style="flex: 1;">Why does my report on usage_events scan every file even though it filters on a single account_key? What layout change would let it prune files?</span>
    <button onclick="var text=document.getElementById('genie-query').innerText; var ta=document.createElement('textarea'); ta.value=text; ta.style.position='fixed'; ta.style.opacity='0'; document.body.appendChild(ta); ta.select(); document.execCommand('copy'); document.body.removeChild(ta); var b=this; b.innerText='Copied!'; setTimeout(function(){ b.innerText='Copy'; }, 2000);" style="background: #FF5F46; color: white; border: none; border-radius: 4px; padding: 6px 12px; font-size: 14pt; cursor: pointer; white-space: nowrap;">Copy</button>
  </div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>
<summary>See response</summary>

![Genie Code response](https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/lab/Lab 2 - Genie Response.png)

</details>
</div>

## E. Fix the Layout
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-steps { display: flex; flex-direction: column; gap: 22px; }
.ds-step { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 18px 20px; border-top: 6px solid var(--accent, #4299E0); }
.blue { --accent:#4299E0; --tint:rgba(66,153,224,0.10); }
.green { --accent:#00A972; --tint:rgba(0,169,114,0.10); }
.amber { --accent:#FFAB00; --tint:rgba(255,171,0,0.12); }
.coral { --accent:#FF5F46; --tint:rgba(255,95,70,0.10); }
.ds-step-head { display: flex; align-items: center; gap: 12px; margin-bottom: 8px; }
.ds-step-num { width: 36px; height: 36px; border-radius: 50%; background: var(--accent); color: #fff; font-size: 18pt; font-weight: 800; line-height: 1; display: flex; align-items: center; justify-content: center; flex-shrink: 0; }
.ds-step-title { font-size: 18pt; font-weight: 700; color: #0b2026; }
.ds-step-text { font-size: 14pt; color: #0b2026; line-height: 1.6; }
.ds-step-text ul { margin: 0; padding-left: 20px; }
.ds-step-text li { margin-bottom: 8px; }
.ds-step-text code { background: #EEEDE9; padding: 2px 6px; border-radius: 3px; font-size: 14pt; color: #0b2026; font-family: Menlo, Consolas, monospace; }
.ds-step-summary { padding: 16px 18px; background: #FFF6F4; border: 2px solid #FF5F46; border-radius: 10px; }
.ds-step-summary-title { font-size: 16pt; font-weight: 800; color: #FF5F46; margin-bottom: 8px; }
.ds-step-summary-text { font-size: 15pt; line-height: 1.6; color: #0b2026; }
</style>

<div style="font-size:15pt;line-height:1.6;margin-bottom:18px;">Write the following in the <strong>next cell</strong>. The report answer stays identical; only the layout changes.</div>

<div class="ds-steps">
<div class="ds-step green">
  <div class="ds-step-head"><div class="ds-step-num">1</div><div class="ds-step-title">Recreate with CTAS and CLUSTER BY</div></div>
  <div class="ds-step-text">
    <ul>
      <li>CTAS so both tables exist side by side for the benchmark. Cluster by the column the report <strong>filters</strong> on, not the one it was partitioned on.</li>
    </ul>
    <div style="margin-top:10px;border-radius:6px;overflow:hidden;border:1px solid #DCE0E2;"><div style="display:flex;justify-content:flex-end;background:#EEEDE9;padding:6px 12px;"><button onclick="var pre=this.parentElement.nextElementSibling;var ta=document.createElement('textarea');ta.value=pre.innerText;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.select();document.execCommand('copy');document.body.removeChild(ta);var b=this;b.innerText='\u2713 Copied!';setTimeout(function(){b.innerText='Copy';},2000);" style="background:#FF5F46;color:#FFFFFF;border:none;border-radius:6px;padding:6px 14px;font-size:14pt;font-weight:700;cursor:pointer;white-space:nowrap;">Copy</button></div><pre style="margin:0;padding:14px;background:#F9F7F4;font-family:Menlo,Consolas,monospace;font-size:14pt;line-height:1.6;color:#0b2026;overflow-x:auto;white-space:pre-wrap;">spark.sql(f"DROP TABLE IF EXISTS {CLUSTERED_TABLE}")
spark.sql(f"""
  CREATE TABLE {CLUSTERED_TABLE}
  CLUSTER BY (account_key)
  AS SELECT * FROM {PARTITIONED_TABLE}
""")</pre></div>
  </div>
</div>
<div class="ds-step amber">
  <div class="ds-step-head"><div class="ds-step-num">2</div><div class="ds-step-title">Set a small file size and OPTIMIZE FULL</div></div>
  <div class="ds-step-text">
    <ul>
      <li>Small <code>targetFileSize</code> so OPTIMIZE produces several narrow account bands and file pruning is visible at lab scale. <code>OPTIMIZE FULL</code> clusters every existing row (regular OPTIMIZE is incremental).</li>
    </ul>
    <div style="margin-top:10px;border-radius:6px;overflow:hidden;border:1px solid #DCE0E2;"><div style="display:flex;justify-content:flex-end;background:#EEEDE9;padding:6px 12px;"><button onclick="var pre=this.parentElement.nextElementSibling;var ta=document.createElement('textarea');ta.value=pre.innerText;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.select();document.execCommand('copy');document.body.removeChild(ta);var b=this;b.innerText='\u2713 Copied!';setTimeout(function(){b.innerText='Copy';},2000);" style="background:#FF5F46;color:#FFFFFF;border:none;border-radius:6px;padding:6px 14px;font-size:14pt;font-weight:700;cursor:pointer;white-space:nowrap;">Copy</button></div><pre style="margin:0;padding:14px;background:#F9F7F4;font-family:Menlo,Consolas,monospace;font-size:14pt;line-height:1.6;color:#0b2026;overflow-x:auto;white-space:pre-wrap;">spark.sql(f"ALTER TABLE {CLUSTERED_TABLE} SET TBLPROPERTIES ('delta.targetFileSize' = '16mb')")
spark.sql(f"OPTIMIZE {CLUSTERED_TABLE} FULL")</pre></div>
  </div>
</div>
<div class="ds-step blue">
  <div class="ds-step-head"><div class="ds-step-num">3</div><div class="ds-step-title">Collect statistics</div></div>
  <div class="ds-step-text">
    <ul>
      <li>Equalize stats so the only variable is the layout. The setup already analyzed the inherited table.</li>
    </ul>
    <div style="margin-top:10px;border-radius:6px;overflow:hidden;border:1px solid #DCE0E2;"><div style="display:flex;justify-content:flex-end;background:#EEEDE9;padding:6px 12px;"><button onclick="var pre=this.parentElement.nextElementSibling;var ta=document.createElement('textarea');ta.value=pre.innerText;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.select();document.execCommand('copy');document.body.removeChild(ta);var b=this;b.innerText='\u2713 Copied!';setTimeout(function(){b.innerText='Copy';},2000);" style="background:#FF5F46;color:#FFFFFF;border:none;border-radius:6px;padding:6px 14px;font-size:14pt;font-weight:700;cursor:pointer;white-space:nowrap;">Copy</button></div><pre style="margin:0;padding:14px;background:#F9F7F4;font-family:Menlo,Consolas,monospace;font-size:14pt;line-height:1.6;color:#0b2026;overflow-x:auto;white-space:pre-wrap;">spark.sql(f"ANALYZE TABLE {CLUSTERED_TABLE} COMPUTE STATISTICS FOR ALL COLUMNS")
spark.sql(f"DESCRIBE DETAIL {CLUSTERED_TABLE}").select(
    "clusteringColumns", "partitionColumns", "numFiles").display()</pre></div>
  </div>
</div>
<div class="ds-step-summary">
  <div class="ds-step-summary-title">What success looks like</div>
  <div class="ds-step-summary-text"><code>clusteringColumns</code> shows <code>[account_key]</code>, <code>partitionColumns</code> is empty, and <code>numFiles</code> is in the single digits. Use the Copy buttons above if you get stuck.</div>
</div>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Why CTAS instead of ALTER TABLE
- On serverless you can convert in place with `ALTER TABLE ... REPLACE PARTITIONED BY WITH CLUSTER BY (...)`. But here we use CTAS so both the partitioned and clustered tables coexist for the Section F benchmark.
- One clustering key is right: the report filters on one column (`account_key`). Up to four keys are allowed, but fewer is better.

### Why the small targetFileSize
- At lab scale (184 MB) the default target would produce one or two files, so pruning would be invisible. Setting 16 MB creates several narrow account bands, making the pruning delta obvious in the profile.
- At production scale, file-size autotuning handles this for you.

### Why ANALYZE
- `ANALYZE TABLE ... COMPUTE STATISTICS FOR ALL COLUMNS` ensures both tables carry equivalent column stats so the comparison is fair. Predictive Optimization runs ANALYZE asynchronously on managed tables, but we need it immediately for the benchmark.

</details>

In [ ]:
# TODO: build the clustered table.
# Source: PARTITIONED_TABLE -> Target: CLUSTERED_TABLE

## F. Benchmark

Re-benchmark the report on your clustered table with the same method, then confirm the improvement is structural.

**Your task:**
1. Run the benchmark cell below. It reruns the report on the clustered table and reports a steady-state median.
2. Run the proof cell after. It compares the layouts with `DESCRIBE DETAIL` and pulls `read_files`, `pruned_files`, `read_bytes`, and `total_task_duration_ms` from `system_vw.query.history`.
3. `pruned_files` should now be above 0 and `read_bytes` and `total_task_duration_ms` should drop. Fewer bytes read means fewer DBUs per run.

In [ ]:
# Guard: this cell needs the table you build in Section E.
_clustered_name = f"{my_catalog}.{LAB_SCHEMA}.usage_events_clustered"
if not spark.catalog.tableExists(_clustered_name):
    print("usage_events_clustered does not exist yet.")
    print("Complete Section E first, then re-run this cell.")
else:
    # Capture the boundary timestamp before the after-report runs.
    # prove_pruning() uses it to label before vs after in system_vw.query.history.
    from datetime import datetime, timezone
    split_time = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%S')

    after_report_df = spark.sql(AFTER_REPORT_SQL)
    after_report_rows = after_report_df.count()
    print(f"Clustered report returned {after_report_rows:,} grouped rows (same answer as before).\n")

    median_after = benchmark_query(AFTER_REPORT_SQL, "after (clustered)")

    print("\n" + "-" * 60)
    print(f"BEFORE (partitioned) median: {median_before:.2f}s")
    print(f"AFTER  (clustered)   median: {median_after:.2f}s")
    if median_after > 0:
        print(f"Wall-clock speedup: {median_before / median_after:.2f}x  (confirm the real win with the profile metrics below)")

In [ ]:
# Physical layout proof (lag-free): before is partitioned with many files, after is clustered. 
# clusteringColumns should show your key, partitionColumns should be empty.
print("=== Physical layout (DESCRIBE DETAIL) ===")
measure_layout(PARTITIONED_TABLE, "BEFORE (partitioned)")
if spark.catalog.tableExists(f"{my_catalog}.{LAB_SCHEMA}.usage_events_clustered"):
    measure_layout(CLUSTERED_TABLE, "AFTER  (clustered)")
else:
    print("AFTER  (clustered): not built yet. Complete Section E first.")

print("\n=== Profile metrics (system_vw.query.history) ===")
# read_files vs pruned_files (pruning), read_bytes (data scanned), and
# total_task_duration_ms (the serverless cost proxy) for the before and after runs.
if 'split_time' not in globals():
    print("split_time not set. Run the benchmark cell above first.")
else:
    prove_pruning(split_time=split_time)

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### What the proof should show
- **`pruned_files` rises above 0.** This is the headline. The clustered table co-locates one account's rows into a few files, and the small target file size gives OPTIMIZE several narrow account bands to work with. The scan skips the files whose `account_key` range excludes the target account. The partitioned table could not prune a single file.
- **`read_bytes` drops.** Fewer files opened and row-group skipping inside them mean the scan touches far less data to answer the same question.
- **`total_task_duration_ms` drops.** This is the cost proxy on serverless: DBUs scale with aggregated task time. Fewer bytes read means fewer, shorter tasks, which means fewer DBUs per run. Multiply that per-run saving by a daily report and the savings compound.
- **Wall-clock is the least reliable signal.** It is inflated by disk-cache warming for whichever query ran first. Trust `total_task_duration_ms` and `pruned_files` over the clock.
- **`shuffle_read_bytes` stays similar.** You did not change the join or aggregation, only the scan. The win is in reading less, which is exactly the lever this part of the course targets.
- **Ingestion lag.** `system_vw.query.history` lags 5 to 10 minutes. If a row is missing, wait and re-run the proof cell. The physical `DESCRIBE DETAIL` comparison is immediate.

</details>

## Key Takeaway
<br />
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-band-solid { background: #1B5162; color: #fff; border-radius: 10px; padding: 22px 28px; text-align: center; border-top: 6px solid var(--accent, #FF5F46); box-shadow: 0 2px 10px rgba(27,49,57,0.12); margin-bottom: 20px; }
.ds-band-lead { font-size: 17pt; font-weight: 700; color: #fff; line-height: 1.3; margin-bottom: 8px; }
.ds-band-sub { font-size: 14pt; color: #F9F7F4; font-weight: 600; line-height: 1.4; }
.ds-callout { border-left: 6px solid var(--accent, #1B5162); border-radius: 8px; padding: 18px 24px; background: var(--tint, #F8F9FC); box-shadow: 0 2px 8px rgba(27,49,57,0.06); font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-eyebrow { font-size: 14pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: var(--accent, #1B5162); margin-bottom: 6px; }
.ds-callout ul { margin: 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-callout li { margin-bottom: 10px; } .ds-callout li:last-child { margin-bottom: 0; }
.caution { --accent:#FFAB00; --tint:rgba(255,171,0,0.12); }
.ds-callout-title { font-size: 16pt; font-weight: 700; color: #0b2026; margin-bottom: 10px; line-height: 1.3; }
</style>

<div class="ds-band-solid">
  <div class="ds-band-lead">A layout that doesn't match the query's filter columns forces a full scan, no matter how many files exist.</div>
  <div class="ds-band-sub">Measure the inherited layout, identify the mismatch, recluster on the filter column and prove fewer bytes were read.</div>
</div>
</div>

## Conclusion

In this lab, you fixed an inherited slow report end to end:

1. **Measured first**: ran the report, opened its Query Profile and recorded an honest baseline with the median wall time and the scan metrics.
2. **Diagnosed the root cause**: over-partitioning by `event_date` on a table the report filters by `account_key`, giving zero partition pruning and no file skipping.
3. **Re-laid out the data**: recreated the table with liquid clustering via CTAS (so both layouts exist for comparison), clustered on the filter column, ran `OPTIMIZE FULL` and collected statistics.
4. **Re-benchmarked and proved the win**: `pruned_files` rose above 0, `read_bytes` and `total_task_duration_ms` dropped, and fewer bytes read means fewer DBUs per run.

You also saw that Genie Code could reach the same conclusion, and because you diagnosed it yourself first, you can judge whether its answer is right.


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>